# Semana 8: análise exploratória E3, distância à borda do rótulo

Roda `analysis/exploratoria/e3_borda.py`, que segue o E3 de `prereg/plano_exploratorio.md`. **É exploratório**: não aplica categorias do pré-registro e não substitui nada da rodada confirmatória.

Pergunta: quanto da relação entre a taxa de conjunto vazio e a fração de água (RQ3) acompanha, na verdade, a quantidade de borda água-terra.

- **Passo 1 (distâncias).** Relê os rótulos originais (Sen1Floods11 do bucket público; cenas e janelas do WorldFloods v2 no Drive) e calcula, para cada pixel pontuado, a distância até o pixel mais próximo da outra classe. Para antes de gravar se as unidades, os tamanhos e os rótulos não forem iguais aos de `labels_<conjunto>.npz`, ou se os pixels a até 2 de distância não forem exatamente os marcados como borda na inferência.
- **Passo 2 (análise).** Por faixa de distância (1-2, 3-5, 6-10, 11-20, mais de 20): taxa de erro, conjunto vazio, erros não sinalizados e AUROC_e, por regime e conjunto. Por unidade: Spearman entre conjunto vazio e fração de água, a correlação parcial dada a parcela de borda, e a mesma correlação só com os pixels a mais de 10 da borda. Antes de gravar, confere τ e o Spearman do RQ3 contra a rodada confirmatória.

| Célula | O que faz |
|---|---|
| 1 | Instala o `rasterio` |
| 2 | Monta o Drive e confere RAM, disco e pastas |
| 3 | Clona o repositório e confere os scripts congelados |
| 4 | Copia as margens para o disco local e confere pelo manifesto (pula o que já foi copiado) |
| 5 | Baixa os rótulos do Sen1Floods11 e confere cada um pelos hashes gravados na semana 6 |
| 6 | Passo 1: distâncias (cerca de 15 minutos; lê as janelas do Drive) |
| 7 | Passo 2: análise (cerca de 30 minutos) |
| 8 | Copia a saída e o log para `resultados/semana8/e3` |

Só CPU, não precisa de GPU. Use um ambiente com pelo menos 8 GB de RAM. Se alguma célula der erro, pare e me mande a saída inteira.

Depois da célula 8, me mande de `resultados/semana8/e3`: a pasta `tabelas`, a pasta `figuras` e `e3.log`.

In [ ]:
!pip install -q "rasterio>=1.4,<1.6"
from importlib.metadata import version
print('rasterio', version('rasterio'))

In [ ]:
import os, re, json, time, hashlib, shutil, subprocess, sys, csv, urllib.request
import concurrent.futures as cf
import numpy as np
import psutil

from google.colab import drive
drive.mount('/content/drive')
RAIZ = '/content/drive/MyDrive/Mestrado'
assert os.path.isdir(f'{RAIZ}/modelo_fixado'), 'Pasta Mestrado/modelo_fixado não encontrada. Confira se o Colab está na conta do mestrado.'

MARGENS_DRIVE = f'{RAIZ}/resultados/semana6/margens'
CHAVE = f'{RAIZ}/resultados/semana6_chave/chave_codigos.json'
CONFIRMATORIA = f'{RAIZ}/resultados/semana6/analise'
HASHES_S1F11 = f'{RAIZ}/resultados/semana6/entradas_sen1floods11_sha256.csv'
WF2 = f'{RAIZ}/dados/worldfloods_v2'
JANELAS = f'{WF2}/janelas'
SAIDA_DRIVE = f'{RAIZ}/resultados/semana8/e3'
for caminho in (MARGENS_DRIVE, f'{WF2}/test/gt', f'{JANELAS}/label'):
    assert os.path.isdir(caminho), f'Pasta não encontrada: {caminho}'
for caminho in (CHAVE, f'{CONFIRMATORIA}/resultados.json', HASHES_S1F11):
    assert os.path.isfile(caminho), f'Arquivo não encontrado: {caminho}'
assert not (os.path.isdir(SAIDA_DRIVE) and os.listdir(SAIDA_DRIVE)), (
    'A pasta resultados/semana8/e3 já tem arquivos. Não sobrescrevo: me avise antes de rodar de novo.')

def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with open(caminho, 'rb') as f:
        for bloco in iter(lambda: f.read(1 << 24), b''):
            h.update(bloco)
    return h.hexdigest()

ram = psutil.virtual_memory().total / 1e9
disco = shutil.disk_usage('/content')[2] / 1e9
print(f'RAM da sessão: {ram:.1f} GB. Disco local livre: {disco:.0f} GB.')
assert ram >= 8, 'RAM abaixo de 8 GB. Use um ambiente com mais memória.'
assert disco > 30, 'Disco local com menos de 30 GB livres.'

In [ ]:
REPO_URL = 'https://github.com/RoseBorges44/prithvi-confidence-shift'
REPO = '/content/repo'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
commit = subprocess.run(['git', '-C', REPO, 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()

tabela = open(f'{REPO}/analysis/README.md', encoding='utf-8').read()
registrados = dict(re.findall(r'\| `([a-z_]+\.py)` \| `([0-9a-f]{64})` \|', tabela))
assert set(registrados) == {'analise_congelada.py', 'nucleo.py', 'reamostragem.py', 'categorias.py'}, registrados.keys()
for nome, sha in registrados.items():
    assert sha256_arquivo(f'{REPO}/analysis/{nome}') == sha, f'{nome} é diferente do congelado. Não rode.'
data_plano = subprocess.run(['git', '-C', REPO, 'log', '--diff-filter=A', '-1', '--format=%cI %h', '--',
                             'prereg/plano_exploratorio.md'], capture_output=True, text=True, check=True).stdout.strip()
assert data_plano, 'prereg/plano_exploratorio.md não está no repositório.'
SHA_E3 = sha256_arquivo(f'{REPO}/analysis/exploratoria/e3_borda.py')
print(f'Repositório no commit {commit[:7]}. Scripts congelados conferem. Plano adicionado em {data_plano}.')

In [ ]:
LOCAL = '/content/margens'
os.makedirs(LOCAL, exist_ok=True)
manifesto = list(csv.DictReader(open(f'{MARGENS_DRIVE}/manifesto_sha256.csv', newline='')))
assert len(manifesto) == 64, f'O manifesto tem {len(manifesto)} arquivos, esperado 64 (60 execuções e 4 rótulos).'
t0 = time.time()
for linha in manifesto:
    destino = f'{LOCAL}/{linha["arquivo"]}'
    if not (os.path.exists(destino) and os.path.getsize(destino) == int(linha['bytes'])):
        shutil.copy(f'{MARGENS_DRIVE}/{linha["arquivo"]}', destino + '.parcial')
        os.replace(destino + '.parcial', destino)
    assert sha256_arquivo(destino) == linha['sha256'], f'{linha["arquivo"]} diferente do manifesto.'
print(f'{len(manifesto)} arquivos copiados e conferidos pelo manifesto em {(time.time() - t0) / 60:.1f} min.')

In [ ]:
BUCKET = 'https://storage.googleapis.com/sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/'
LB = '/content/sen1floods11_rotulos'
os.makedirs(LB, exist_ok=True)
hashes = {l['arquivo']: l['sha256'] for l in csv.DictReader(open(HASHES_S1F11, newline=''))}
chaves = [str(c) for conj in ('val', 'teste', 'bolivia')
          for c in np.load(f'{LOCAL}/labels_{conj}.npz', allow_pickle=False)['chaves']]
assert len(chaves) == 194 and len(set(chaves)) == 194, len(chaves)

def baixar(chave):
    nome = f'{chave}_LabelHand.tif'
    destino = f'{LB}/{nome}'
    if not (os.path.exists(destino) and sha256_arquivo(destino) == hashes[nome]):
        for tentativa in range(4):
            try:
                urllib.request.urlretrieve(BUCKET + nome, destino + '.parcial')
                os.replace(destino + '.parcial', destino)
                break
            except Exception:
                time.sleep(2 * (tentativa + 1))
    assert sha256_arquivo(destino) == hashes[nome], f'{nome} diferente do hash gravado na semana 6.'
    return nome

with cf.ThreadPoolExecutor(16) as pool:
    feitos = list(pool.map(baixar, chaves))
print(f'{len(feitos)} rótulos do Sen1Floods11 conferidos pelos hashes da semana 6.')

In [ ]:
# Passo 1: distâncias (cerca de 15 minutos)
DIST = '/content/e3_distancias'
LOG = '/content/e3.log'
if os.path.isdir(DIST):
    shutil.rmtree(DIST)   # saída local de uma tentativa interrompida
cmd = (f'cd {REPO} && set -o pipefail && python analysis/exploratoria/e3_borda.py distancias --repo . '
       f'--margens {LOCAL} --s1f11-rotulos {LB} --janelas "{JANELAS}" --wf2 "{WF2}" --saida {DIST} 2>&1 | tee {LOG}')
get_ipython().system(f"bash -c '{cmd}'")
assert os.path.isfile(f'{DIST}/distancias.json'), 'O passo 1 não terminou. Guarde a saída e me mande.'

In [ ]:
# Passo 2: análise (cerca de 30 minutos). Deixe a aba aberta.
SAIDA = '/content/e3'
if os.path.isdir(SAIDA):
    shutil.rmtree(SAIDA)   # saída local de uma tentativa interrompida
cmd = (f'cd {REPO} && set -o pipefail && python analysis/exploratoria/e3_borda.py analise --repo . --margens {LOCAL} '
       f'--chave "{CHAVE}" --confirmatoria "{CONFIRMATORIA}" --distancias {DIST} --saida {SAIDA} 2>&1 | tee -a {LOG}')
get_ipython().system(f"bash -c '{cmd}'")

In [ ]:
assert os.path.isfile(f'{SAIDA}/e3.json'), 'A análise não terminou: não há e3.json. Guarde o log e me mande.'
os.makedirs(SAIDA_DRIVE, exist_ok=True)
shutil.copytree(SAIDA, SAIDA_DRIVE, dirs_exist_ok=True)
shutil.copytree(DIST, f'{SAIDA_DRIVE}/distancias', dirs_exist_ok=True)
shutil.copy(LOG, f'{SAIDA_DRIVE}/e3.log')
INFO = {'commit_repositorio': commit, 'plano': 'prereg/plano_exploratorio.md, E3', 'plano_adicionado': data_plano,
        'e3_sha256': SHA_E3, 'congelados_sha256': registrados, 'rasterio': version('rasterio'),
        'gerado_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
json.dump(INFO, open(f'{SAIDA_DRIVE}/rodada.json', 'w'), indent=2)
for raiz, _, arquivos in os.walk(SAIDA_DRIVE):
    for a in sorted(arquivos):
        print(os.path.relpath(os.path.join(raiz, a), SAIDA_DRIVE))
print(f'\nSaída copiada para {SAIDA_DRIVE}.')